In [1]:
import warnings
warnings.filterwarnings('ignore')

import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

In [2]:
train_df=pd.read_parquet('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/interim/final_train.parquet')
test_df=pd.read_parquet('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/interim/final_test.parquet')

In [3]:

# ============================================================
# CONFIG
# ============================================================

TARGET_COL = 'TARGET'
ID_COL = 'SK_ID_CURR'

N_SPLITS = 10
RANDOM_STATE = 42

In [ ]:
top_features_750=['ORGANIZATION_TYPE',
 'credit_to_annuity',
 'goods_to_credit',
 'EXT_SOURCE_3',
 'EXT_SOURCE_1',
 'DAYS_LAST_PHONE_CHANGE',
 'AMT_ANNUITY',
 'ext_std',
 'OCCUPATION_TYPE',
 'annuity_to_income',
 'EXT_SOURCE_2',
 'DAYS_ID_PUBLISH',
 'ext_mean_x_employment',
 'id_publish_age_ratio',
 'ext_mean',
 'ext_min',
 'DAYS_REGISTRATION',
 'DAYS_BIRTH',
 'REGION_POPULATION_RELATIVE',
 'annuity_per_person',
 'registration_age_ratio',
 'income_to_days',
 'employment_age_ratio',
 'income_to_age',
 'ext_mean_x_age',
 'OWN_CAR_AGE',
 'credit_per_person',
 'ext_mul',
 'ext_max',
 'ext_med',
 'ext_mean_x_income',
 'income_to_price',
 'AMT_GOODS_PRICE',
 'total_consumer_credit_debt',
 'age_years',
 'AMT_CREDIT',
 'income_per_person',
 'total_credit_card_debt',
 'ACTIVE_BUREAU_B_TOTAL_TENURE_MIN',
 'credit_to_income',
 'ACTIVE_BUREAU_B_LEFT_TO_FULL_MEAN',
 'ACTIVE_BUREAU_DAYS_CREDIT_ENDDATE_MIN',
 'BUREAU_B_LEFT_TO_FULL_STD',
 'ACTIVE_BUREAU_B_LEFT_TO_FULL_MAX',
 'BUREAU_B_UPDATE_TO_AGE_STD',
 'DAYS_EMPLOYED',
 'BUREAU_B_PAID_TO_DAYS_MEAN',
 'BUREAU_B_LEFT_TO_FULL_MAX',
 'ext_range',
 'CLOSED_BUREAU_B_UPDATE_TO_AGE_STD',
 'CLOSED_BUREAU_B_UPDATE_TO_AGE_MEAN',
 'BUREAU_B_LOAN_TO_TENURE_MEAN',
 'ACTIVE_BUREAU_B_UPDATE_TO_AGE_MEAN',
 'CLOSED_BUREAU_B_LOAN_TO_TENURE_MEAN',
 'ACTIVE_BUREAU_B_LEFT_TO_FULL_STD',
 'BUREAU_B_UPDATE_TO_AGE_MEAN',
 'BUREAU_B_TOTAL_TENURE_MIN',
 'BUREAU_B_LOAN_TO_TENURE_STD',
 'BUREAU_B_PAID_TO_DAYS_MAX',
 'HOUR_APPR_PROCESS_START',
 'BUREAU_B_PAID_TO_DAYS_STD',
 'BUREAU_DAYS_CREDIT_ENDDATE_STD',
 'BUREAU_DAYS_CREDIT_STD',
 'BUREAU_B_CREDIT_PER_DAY_STD',
 'BUREAU_B_TOTAL_TENURE_STD',
 'ACTIVE_BUREAU_B_EXP_ANNUITY_MEAN',
 'CLOSED_BUREAU_B_LOAN_TO_TENURE_STD',
 'ACTIVE_BUREAU_DAYS_CREDIT_ENDDATE_MEAN',
 'TOTALAREA_MODE',
 'ACTIVE_BUREAU_B_UPDATE_TO_AGE_MAX',
 'BUREAU_B_LEFT_TO_FULL_MEAN',
 'CLOSED_BUREAU_B_LOAN_TO_TENURE_MAX',
 'BUREAU_B_UPDATE_TO_AGE_MAX',
 'CLOSED_BUREAU_B_UPDATE_TO_AGE_MAX',
 'ACTIVE_BUREAU_B_UPDATE_TO_AGE_STD',
 'ACTIVE_BUREAU_B_PAID_TO_DAYS_MEAN',
 'AMT_INCOME_TOTAL',
 'BUREAU_DAYS_CREDIT_ENDDATE_MEAN',
 'ACTIVE_BUREAU_B_LOAN_TO_TENURE_MEAN',
 'BUREAU_B_EXP_ANNUITY_STD',
 'ACTIVE_BUREAU_DAYS_CREDIT_STD',
 'BUREAU_B_ACTUAL_TENURE_MIN',
 'BUREAU_B_EXP_ANNUITY_MEAN',
 'BUREAU_DAYS_CREDIT_UPDATE_STD',
 'CLOSED_BUREAU_DAYS_CREDIT_STD',
 'BUREAU_B_LOAN_TO_TENURE_MAX',
 'BUREAU_AMT_CREDIT_SUM_STD',
 'BUREAU_AMT_CREDIT_SUM_MEAN',
 'CLOSED_BUREAU_AMT_CREDIT_SUM_MEAN',
 'ACTIVE_BUREAU_DAYS_CREDIT_ENDDATE_MAX',
 'CLOSED_BUREAU_B_TOTAL_TENURE_MIN',
 'ACTIVE_BUREAU_DAYS_CREDIT_ENDDATE_STD',
 'CLOSED_BUREAU_B_PAID_TO_DAYS_MEAN',
 'CLOSED_BUREAU_DAYS_CREDIT_ENDDATE_STD',
 'ACTIVE_BUREAU_DAYS_CREDIT_UPDATE_STD',
 'BUREAU_B_TOTAL_TENURE_MEAN',
 'ACTIVE_BUREAU_B_LOAN_TO_TENURE_MAX',
 'CLOSED_BUREAU_B_EXP_ANNUITY_STD',
 'ACTIVE_BUREAU_B_TOTAL_TENURE_MEAN',
 'ACTIVE_BUREAU_B_LOAN_TO_TENURE_STD',
 'CLOSED_BUREAU_DAYS_CREDIT_UPDATE_MAX',
 'CLOSED_BUREAU_B_TOTAL_TENURE_STD',
 'ACTIVE_BUREAU_B_TOTAL_TENURE_STD',
 'CLOSED_BUREAU_B_EXP_ANNUITY_MEAN',
 'CLOSED_BUREAU_AMT_CREDIT_SUM_STD',
 'CLOSED_BUREAU_DAYS_CREDIT_ENDDATE_MAX',
 'BUREAU_B_CREDIT_PER_DAY_MEAN',
 'BUREAU_B_DEBT_TO_TENURE_STD',
 'CLOSED_BUREAU_B_PAID_TO_DAYS_STD',
 'CLOSED_BUREAU_DAYS_CREDIT_UPDATE_STD',
 'ACTIVE_BUREAU_B_DEBT_TO_TENURE_MEAN',
 'BUREAU_B_ANNUITY_LOAN_RATIO_STD',
 'CLOSED_BUREAU_B_TOTAL_TENURE_MEAN',
 'CLOSED_BUREAU_B_PAID_TO_DAYS_MAX',
 'CLOSED_BUREAU_DAYS_CREDIT_ENDDATE_MEAN',
 'ACTIVE_BUREAU_B_CREDIT_PER_DAY_STD',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_SUM',
 'BUREAU_B_ACTUAL_TENURE_MEAN',
 'CLOSED_BUREAU_B_CREDIT_PER_DAY_MEAN',
 'CLOSED_BUREAU_AMT_CREDIT_SUM_SUM',
 'ACTIVE_BUREAU_DAYS_CREDIT_MEAN',
 'ACTIVE_BUREAU_B_EXP_ANNUITY_SUM',
 'BUREAU_DAYS_CREDIT_ENDDATE_MAX',
 'ACTIVE_BUREAU_B_CREDIT_PER_DAY_MEAN',
 'ACTIVE_BUREAU_DAYS_CREDIT_UPDATE_MEAN',
 'LANDAREA_AVG',
 'BUREAU_B_EXP_ANNUITY_MAX',
 'CLOSED_BUREAU_DAYS_CREDIT_MAX',
 'BUREAU_B_TENURE_GAP_MAX',
 'ACTIVE_BUREAU_B_DEBT_TO_TENURE_STD',
 'BUREAU_DAYS_ENDDATE_FACT_MAX',
 'CODE_GENDER',
 'ORGANIZATION_TYPE_FREQ',
 'BUREAU_B_ACTUAL_TENURE_MAX',
 'BUREAU_B_TENURE_GAP_MEAN',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_STD',
 'BUREAU_DAYS_CREDIT_MAX',
 'employment_years',
 'LIVINGAREA_MODE',
 'BUREAU_B_EXP_ANNUITY_SUM',
 'ACTIVE_BUREAU_B_EXP_ANNUITY_STD',
 'BUREAU_B_ACTUAL_TENURE_STD',
 'ACTIVE_BUREAU_DAYS_CREDIT_MAX',
 'ACTIVE_BUREAU_B_CREDIT_PER_DAY_MAX',
 'ACTIVE_BUREAU_DAYS_CREDIT_UPDATE_MAX',
 'ACTIVE_BUREAU_B_ANNUITY_LOAN_RATIO_STD',
 'CLOSED_BUREAU_AMT_CREDIT_SUM_MAX',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_MEAN',
 'YEARS_BEGINEXPLUATATION_AVG',
 'AMT_REQ_CREDIT_BUREAU_YEAR',
 'CLOSED_BUREAU_B_CREDIT_PER_DAY_STD',
 'LANDAREA_MODE',
 'ACTIVE_BUREAU_B_DEBT_TO_DAYS_STD',
 'BUREAU_B_ANNUITY_IMP_STD',
 'APARTMENTS_MODE',
 'BASEMENTAREA_MODE',
 'ACTIVE_BUREAU_B_PAID_TO_DAYS_STD',
 'BUREAU_B_TOTAL_TENURE_MAX',
 'BUREAU_AMT_CREDIT_MAX_OVERDUE_MEAN',
 'ACTIVE_BUREAU_B_ANNUITY_LOAN_RATIO_MEAN',
 'CLOSED_BUREAU_B_EXP_ANNUITY_MAX',
 'ACTIVE_BUREAU_DAYS_CREDIT_UPDATE_MIN',
 'WEEKDAY_APPR_PROCESS_START',
 'BUREAU_B_CREDIT_PER_DAY_MAX',
 'BUREAU_B_MAX_OVERDUE_TO_CREDIT_STD',
 'BUREAU_B_TENURE_GAP_STD',
 'BUREAU_B_DEBT_TO_TENURE_MEAN',
 'BUREAU_DAYS_CREDIT_MEAN',
 'BUREAU_DAYS_CREDIT_UPDATE_MAX',
 'YEARS_BEGINEXPLUATATION_MODE',
 'BUREAU_AMT_CREDIT_MAX_OVERDUE_MAX',
 'BUREAU_AMT_CREDIT_SUM_SUM',
 'LIVINGAREA_AVG',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_DEBT_MEAN',
 'BUREAU_AMT_CREDIT_SUM_MAX',
 'total_consumer_credit_max_overdue',
 'BUREAU_B_MAX_OVERDUE_MISSING_MEAN',
 'BUREAU_B_MAX_OVERDUE_TO_CREDIT_MEAN',
 'BUREAU_DAYS_ENDDATE_FACT_STD',
 'CLOSED_BUREAU_B_CREDIT_PER_DAY_MAX',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_DEBT_STD',
 'BUREAU_B_ANNUITY_LOAN_RATIO_MEAN',
 'ACTIVE_BUREAU_DAYS_CREDIT_MIN',
 'ACTIVE_BUREAU_B_TOTAL_TENURE_MAX',
 'CLOSED_BUREAU_B_TOTAL_TENURE_MAX',
 'CLOSED_BUREAU_B_EXP_ANNUITY_SUM',
 'BUREAU_DAYS_CREDIT_ENDDATE_MIN',
 'BUREAU_B_DEBT_TO_DAYS_STD',
 'BUREAU_B_MAX_OVERDUE_TO_CREDIT_MAX',
 'BUREAU_AMT_CREDIT_SUM_DEBT_STD',
 'BASEMENTAREA_AVG',
 'APARTMENTS_AVG',
 'ACTIVE_BUREAU_B_DEBT_TO_DAYS_MEAN',
 'consumer_credit_max_overdue_to_typedebt',
 'ACTIVE_BUREAU_B_PAID_TO_DAYS_MAX',
 'BUREAU_B_DEBT_TO_DAYS_MEAN',
 'CLOSED_BUREAU_B_MAX_OVERDUE_MISSING_MEAN',
 'BUREAU_B_EXPIRED_LOAN_MEAN',
 'BUREAU_B_TENURE_GAP_MIN',
 'ACTIVE_BUREAU_B_ANNUITY_IMP_MEAN',
 'ACTIVE_BUREAU_B_EXP_ANNUITY_MAX',
 'CLOSED_BUREAU_B_ACTUAL_TENURE_MEAN',
 'BUREAU_B_ANNUITY_IMP_MEAN',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_MAX',
 'ACTIVE_BUREAU_B_DEBT_TO_TENURE_MAX',
 'NAME_EDUCATION_TYPE',
 'CLOSED_BUREAU_DAYS_CREDIT_ENDDATE_MIN',
 'REGION_RATING_CLIENT_W_CITY',
 'CLOSED_BUREAU_B_ACTUAL_TENURE_STD',
 'NONLIVINGAREA_AVG',
 'CLOSED_BUREAU_DAYS_ENDDATE_FACT_MAX',
 'BUREAU_B_CLOSED_LATE_MEAN',
 'DEF_30_CNT_SOCIAL_CIRCLE',
 'ACTIVE_BUREAU_B_ANNUITY_LOAN_RATIO_MAX',
 'BUREAU_DAYS_CREDIT_UPDATE_MEAN',
 'CLOSED_BUREAU_B_TENURE_GAP_MAX',
 'OCCUPATION_TYPE_FREQ',
 'OBS_30_CNT_SOCIAL_CIRCLE',
 'BUREAU_B_CLOSED_EARLY_MEAN',
 'COMMONAREA_AVG',
 'BUREAU_DAYS_ENDDATE_FACT_MEAN',
 'CLOSED_BUREAU_B_ACTUAL_TENURE_MIN',
 'CLOSED_BUREAU_DAYS_CREDIT_UPDATE_MEAN',
 'NONLIVINGAREA_MODE',
 'COMMONAREA_MODE',
 'CLOSED_BUREAU_B_TENURE_GAP_STD',
 'CLOSED_BUREAU_B_TENURE_GAP_MEAN',
 'CLOSED_BUREAU_B_ACTUAL_TENURE_MAX',
 'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN',
 'ACTIVE_BUREAU_AMT_CREDIT_MAX_OVERDUE_MAX',
 'YEARS_BEGINEXPLUATATION_MEDI',
 'BUREAU_B_DEBT_TO_TENURE_MAX',
 'total_microloan_debt',
 'BUREAU_BB_NUM_MON_STD',
 'BUREAU_B_ANNUITY_LOAN_RATIO_MAX',
 'ACTIVE_BUREAU_B_ANNUITY_IMP_STD',
 'BUREAU_DAYS_CREDIT_MIN',
 'CLOSED_BUREAU_DAYS_CREDIT_MEAN',
 'FLAG_WORK_PHONE',
 'CLOSED_BUREAU_B_CREDIT_AGE_MIN',
 'BASEMENTAREA_MEDI',
 'BUREAU_B_ACTIVE_DEBT_MEAN',
 'NAME_FAMILY_STATUS',
 'CLOSED_BUREAU_B_CLOSED_EARLY_MEAN',
 'ACTIVE_BUREAU_B_CREDIT_AGE_MEAN',
 'CLOSED_BUREAU_B_UPDATE_RECENCY_MIN',
 'CLOSED_BUREAU_B_MAX_OVERDUE_TO_CREDIT_STD',
 'CLOSED_BUREAU_DAYS_CREDIT_MIN',
 'LIVINGAREA_MEDI',
 'total_docs_provided',
 'LIVINGAPARTMENTS_AVG',
 'CLOSED_BUREAU_DAYS_ENDDATE_FACT_STD',
 'BUREAU_DAYS_ENDDATE_FACT_MIN',
 'ACTIVE_BUREAU_B_DEBT_TO_DAYS_MAX',
 'ACTIVE_BUREAU_AMT_CREDIT_MAX_OVERDUE_MEAN',
 'ACTIVE_BUREAU_B_MAX_OVERDUE_MISSING_MEAN',
 'YEARS_BUILD_AVG',
 'BUREAU_B_RECENT_365D_MEAN',
 'BUREAU_BB_NUM_MON_MEAN',
 'BUREAU_B_CREDIT_AGE_MEAN',
 'NONLIVINGAREA_MEDI',
 'OBS_60_CNT_SOCIAL_CIRCLE',
 'LANDAREA_MEDI',
 'CLOSED_BUREAU_AMT_CREDIT_MAX_OVERDUE_MAX',
 'BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN',
 'BUREAU_AMT_CREDIT_SUM_LIMIT_STD',
 'ENTRANCES_AVG',
 'BUREAU_B_DEBT_MISSING_MEAN',
 'LIVINGAPARTMENTS_MODE',
 'BUREAU_B_CREDIT_AGE_MIN',
 'CLOSED_BUREAU_B_ANNUITY_IMP_STD',
 'CLOSED_BUREAU_B_CLOSED_LATE_MEAN',
 'APARTMENTS_MEDI',
 'CLOSED_BUREAU_AMT_CREDIT_MAX_OVERDUE_MEAN',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN',
 'ACTIVE_BUREAU_B_CREDIT_AGE_MIN',
 'ACTIVE_BUREAU_B_CREDIT_AGE_MAX',
 'CLOSED_BUREAU_DAYS_CREDIT_UPDATE_MIN',
 'CLOSED_BUREAU_B_DEBT_MISSING_MEAN',
 'CLOSED_BUREAU_B_MAX_OVERDUE_TO_CREDIT_MEAN',
 'DEF_60_CNT_SOCIAL_CIRCLE',
 'ACTIVE_BUREAU_BB_NUM_MON_MEAN',
 'BUREAU_B_DEBT_TO_DAYS_MAX',
 'BUREAU_DAYS_CREDIT_UPDATE_MIN',
 'CLOSED_BUREAU_BB_NUM_MON_MEAN',
 'FLAG_DOCUMENT_3',
 'total_mortgage_debt',
 'ACTIVE_BUREAU_B_UPDATE_RECENCY_MEAN',
 'ACTIVE_BUREAU_B_MAX_OVERDUE_TO_CREDIT_STD',
 'ACTIVE_BUREAU_B_DEBT_MISSING_SUM',
 'BUREAU_BB_LATEST_MON_MEAN',
 'CLOSED_BUREAU_B_ANNUITY_LOAN_RATIO_STD',
 'BUREAU_BB_OLDEST_MON_STD',
 'CLOSED_BUREAU_B_MAX_OVERDUE_TO_CREDIT_MAX',
 'YEARS_BUILD_MODE',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_LIMIT_STD',
 'total_car_loan_debt',
 'CLOSED_BUREAU_B_EXPIRED_LOAN_MEAN',
 'CLOSED_BUREAU_B_UPDATE_RECENCY_MEAN',
 'total_credit_card_count',
 'CLOSED_BUREAU_B_ANNUITY_LOAN_RATIO_MEAN',
 'AMT_REQ_CREDIT_BUREAU_QRT',
 'BUREAU_B_CREDIT_AGE_MAX',
 'CLOSED_BUREAU_BB_OLDEST_MON_STD',
 'ACTIVE_BUREAU_B_MAX_OVERDUE_TO_CREDIT_MEAN',
 'BUREAU_AMT_CREDIT_SUM_DEBT_MAX',
 'CLOSED_BUREAU_BB_NUM_MON_STD',
 'BUREAU_B_HAS_ACTUAL_END_MEAN',
 'CLOSED_BUREAU_B_TENURE_GAP_MIN',
 'BUREAU_B_UPDATE_RECENCY_MEAN',
 'ACTIVE_BUREAU_B_ANNUITY_IMP_SUM',
 'CLOSED_BUREAU_BB_OLDEST_MON_MAX',
 'CLOSED_BUREAU_DAYS_ENDDATE_FACT_MEAN',
 'BUREAU_BB_OLDEST_MON_MEAN',
 'COMMONAREA_MEDI',
 'BUREAU_BB_OLDEST_MON_MAX',
 'BUREAU_B_ACTIVE_DEBT_SUM',
 'BUREAU_BB_NUM_MON_MAX',
 'ACTIVE_BUREAU_B_MAX_OVERDUE_TO_CREDIT_MAX',
 'ACTIVE_BUREAU_BB_NUM_MON_STD',
 'total_consumer_credit_count',
 'ACTIVE_BUREAU_B_DEBT_MISSING_MEAN',
 'BUREAU_AMT_CREDIT_SUM_DEBT_SUM',
 'ACTIVE_BUREAU_LOAN_COUNT_SUM',
 'CLOSED_BUREAU_B_CREDIT_AGE_MAX',
 'CLOSED_BUREAU_BB_LATEST_MON_STD',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_DEBT_MAX',
 'BUREAU_BB_LATEST_MON_STD',
 'BUREAU_B_ANNUITY_IMP_SUM',
 'CLOSED_BUREAU_B_ANNUITY_IMP_MEAN',
 'CLOSED_BUREAU_BB_LATEST_MON_MEAN',
 'CLOSED_BUREAU_B_CREDIT_AGE_MEAN',
 'ACTIVE_BUREAU_BB_OLDEST_MON_STD',
 'ACTIVE_BUREAU_BB_OLDEST_MON_MEAN',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_DEBT_SUM',
 'BUREAU_BB_SMA0_RATIO_MEAN',
 'FLOORSMAX_AVG',
 'FLOORSMIN_AVG',
 'BUREAU_BB_HISTORY_AGE_MAX',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_LIMIT_MAX',
 'credit_card_max_overdue_to_typedebt',
 'ACTIVE_BUREAU_B_MAX_OVERDUE_MISSING_SUM',
 'LIVINGAPARTMENTS_MEDI',
 'ACTIVE_BUREAU_BB_LATEST_MON_MEAN',
 'CLOSED_BUREAU_B_UPDATE_RECENCY_MAX',
 'BUREAU_B_RECENT_180D_MEAN',
 'ENTRANCES_MODE',
 'CLOSED_BUREAU_DAYS_ENDDATE_FACT_MIN',
 'REG_CITY_NOT_LIVE_CITY',
 'BUREAU_B_ACTIVE_DEBT_MAX',
 'CLOSED_BUREAU_BB_NUM_MON_MAX',
 'CLOSED_BUREAU_B_ANNUITY_LOAN_RATIO_MAX',
 'CLOSED_BUREAU_BB_OLDEST_MON_MEAN',
 'BUREAU_B_MAX_OVERDUE_TO_DEBT_MEAN',
 'ACTIVE_BUREAU_BB_NUM_MON_MAX',
 'BUREAU_AMT_CREDIT_SUM_LIMIT_MAX',
 'BUREAU_B_UPDATE_RECENCY_MAX',
 'ACTIVE_BUREAU_B_UPDATE_RECENCY_MAX',
 'BUREAU_BB_SMA0_RATIO_MAX',
 'BUREAU_B_MAX_OVERDUE_MISSING_SUM',
 'NONLIVINGAPARTMENTS_AVG',
 'ACTIVE_BUREAU_BB_LATEST_MON_STD',
 'ACTIVE_BUREAU_BB_HISTORY_AGE_MAX',
 'NAME_INCOME_TYPE_FREQ',
 'ACTIVE_BUREAU_B_MAX_OVERDUE_TO_DEBT_MEAN',
 'total_credit_card_max_overdue',
 'ACTIVE_BUREAU_B_RECENT_365D_SUM',
 'ACTIVE_BUREAU_BB_MISSING_SUM',
 'ACTIVE_BUREAU_BB_OLDEST_MON_MAX',
 'ELEVATORS_AVG',
 'BUREAU_B_ANNUITY_IMP_MAX',
 'CLOSED_BUREAU_BB_HISTORY_AGE_MAX',
 'BUREAU_B_UPDATE_RECENCY_MIN',
 'BUREAU_B_MAX_OVERDUE_TO_DEBT_MAX',
 'BUREAU_LOAN_COUNT_SUM',
 'CLOSED_BUREAU_BB_SMA0_RATIO_MEAN',
 'ACTIVE_BUREAU_B_ANNUITY_IMP_MAX',
 'ACTIVE_BUREAU_B_UPDATE_RECENCY_MIN',
 'CLOSED_BUREAU_B_RECENT_365D_MEAN',
 'NAME_CONTRACT_TYPE',
 'CLOSED_BUREAU_B_MAX_OVERDUE_MISSING_SUM',
 'ACTIVE_BUREAU_B_RECENT_365D_MEAN',
 'CLOSED_BUREAU_B_EXPIRED_LOAN_SUM',
 'ENTRANCES_MEDI',
 'BUREAU_BB_SMA0_MEAN',
 'BUREAU_BB_MEAN_DPD_MAX',
 'BUREAU_B_ANNUITY_MISSING_MEAN',
 'BUREAU_B_ANNUITY_MISSING_SUM',
 'CLOSED_BUREAU_B_ANNUITY_IMP_SUM',
 'CNT_FAM_MEMBERS',
 'YEARS_BUILD_MEDI',
 'BUREAU_BB_MEAN_DPD_MEAN',
 'address_mismatch_count',
 'NAME_HOUSING_TYPE',
 'CLOSED_BUREAU_B_ANNUITY_IMP_MAX',
 'contact_count',
 'WALLSMATERIAL_MODE',
 'BUREAU_BB_SMA0_RATIO_STD',
 'ACTIVE_BUREAU_B_MAX_OVERDUE_TO_DEBT_MAX',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_LIMIT_SUM',
 'BUREAU_BB_MISSING_SUM',
 'ACTIVE_BUREAU_B_MAX_OVERDUE_TO_DEBT_STD',
 'CLOSED_BUREAU_BB_SMA0_RATIO_MAX',
 'BUREAU_B_MAX_OVERDUE_TO_DEBT_STD',
 'CLOSED_BUREAU_BB_MISSING_SUM',
 'BUREAU_B_RECENT_90D_MEAN',
 'ACTIVE_BUREAU_B_ANNUITY_MISSING_SUM',
 'BUREAU_B_RECENT_365D_SUM',
 'CLOSED_BUREAU_B_ANNUITY_MISSING_SUM',
 'BUREAU_AMT_CREDIT_SUM_LIMIT_SUM',
 'ACTIVE_BUREAU_B_DEBT_TO_CREDIT_MEAN',
 'REGION_RATING_CLIENT',
 'CLOSED_BUREAU_B_ANNUITY_MISSING_MEAN',
 'BUREAU_BB_MAX_DEFAULT_MEAN',
 'child_ratio',
 'NONLIVINGAPARTMENTS_MODE',
 'NAME_TYPE_SUITE',
 'BUREAU_B_EXPIRED_LOAN_SUM',
 'FLOORSMIN_MODE',
 'CLOSED_BUREAU_BB_MEAN_DPD_MEAN',
 'ACTIVE_BUREAU_B_DEBT_TO_CREDIT_MAX',
 'ACTIVE_BUREAU_B_REMAINING_TERM_MEAN',
 'CLOSED_BUREAU_BB_MEAN_DPD_MAX',
 'ACTIVE_BUREAU_B_RECENT_180D_MEAN',
 'ACTIVE_BUREAU_B_EXPIRED_LOAN_MEAN',
 'ACTIVE_BUREAU_B_DEBT_TO_CREDIT_STD',
 'ACTIVE_BUREAU_B_REMAINING_TERM_MIN',
 'BUREAU_B_DEBT_MISSING_SUM',
 'NAME_INCOME_TYPE',
 'CNT_CHILDREN',
 'BUREAU_B_CREDIT_AGE_STD',
 'CLOSED_BUREAU_BB_SMA0_RATIO_STD',
 'BUREAU_B_DEBT_TO_CREDIT_MAX',
 'total_microloan_count',
 'BUREAU_B_DEBT_TO_CREDIT_STD',
 'BUREAU_B_REMAINING_TERM_STD',
 'FLAG_OWN_REALTY',
 'BUREAU_BB_HISTORY_AGE_MEAN',
 'ELEVATORS_MODE',
 'BUREAU_IS_SOLD_MEAN',
 'BUREAU_B_DEBT_TO_CREDIT_MEAN',
 'AMT_REQ_CREDIT_BUREAU_MON',
 'ACTIVE_BUREAU_B_CREDIT_AGE_STD',
 'ACTIVE_BUREAU_B_REMAINING_TERM_STD',
 'NONLIVINGAPARTMENTS_MEDI',
 'FLOORSMAX_MODE',
 'FLAG_PHONE',
 'adult_count',
 'BUREAU_B_HAS_ACTUAL_END_SUM',
 'CLOSED_BUREAU_LOAN_COUNT_SUM',
 'BUREAU_B_UPDATE_RECENCY_STD',
 'CLOSED_BUREAU_BB_MAX_DEFAULT_MEAN',
 'BUREAU_BB_SMA0_MAX',
 'BUREAU_B_REMAINING_TERM_MEAN',
 'BUREAU_B_REMAINING_TERM_MAX',
 'CLOSED_BUREAU_B_DEBT_MISSING_SUM',
 'ACTIVE_BUREAU_B_UPDATE_RECENCY_STD',
 'ACTIVE_BUREAU_BB_SMA0_RATIO_STD',
 'CLOSED_BUREAU_BB_HISTORY_AGE_MEAN',
 'CLOSED_BUREAU_B_REMAINING_TERM_MEAN',
 'CLOSED_BUREAU_B_CREDIT_AGE_STD',
 'BUREAU_B_CLOSED_EARLY_SUM',
 'FLOORSMAX_MEDI',
 'CLOSED_BUREAU_B_REMAINING_TERM_STD',
 'housing_missing_count',
 'CLOSED_BUREAU_BB_SMA0_MEAN',
 'ACTIVE_BUREAU_B_DEBT_PER_DAY_STD',
 'OCCUPATION_SKILL_GROUP',
 'CODE_GENDER_BIN',
 'BUREAU_B_RECENT_180D_SUM',
 'ACTIVE_BUREAU_BB_HISTORY_AGE_MEAN',
 'CLOSED_BUREAU_B_CLOSED_EARLY_SUM',
 'FLAG_OWN_CAR',
 'CLOSED_BUREAU_BB_LATEST_MON_MAX',
 'ACTIVE_BUREAU_B_REMAINING_TERM_MAX',
 'ACTIVE_BUREAU_BB_SMA0_RATIO_MAX',
 'ACTIVE_BUREAU_B_ANNUITY_MISSING_MEAN',
 'LIVE_CITY_NOT_WORK_CITY',
 'ACTIVE_BUREAU_BB_SMA0_MAX',
 'ELEVATORS_MEDI',
 'FLOORSMIN_MEDI',
 'CLOSED_BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN',
 'CLOSED_BUREAU_B_UPDATE_RECENCY_STD',
 'CLOSED_BUREAU_B_REMAINING_TERM_MAX',
 'ACTIVE_BUREAU_B_DEBT_X_PROLONGED_MEAN',
 'ACTIVE_BUREAU_BB_SMA0_RATIO_MEAN',
 'BUREAU_B_LOG_CURR_DPD_STD',
 'CLOSED_BUREAU_B_HAS_OVERDUE_MEAN',
 'ACTIVE_BUREAU_BB_MEAN_DPD_MAX',
 'ACTIVE_BUREAU_B_CREDIT_X_PROLONGED_MEAN',
 'ACTIVE_BUREAU_BB_MEAN_DPD_MEAN',
 'BUREAU_BB_SMA0_SUM',
 'ACTIVE_BUREAU_B_RECENT_90D_MEAN',
 'consumer_credit_max_overdue_to_total',
 'ACTIVE_BUREAU_B_RECENT_180D_SUM',
 'missing_contact_count',
 'CLOSED_BUREAU_AMT_CREDIT_SUM_LIMIT_STD',
 'ACTIVE_BUREAU_B_ACTIVE_DEBT_MEAN',
 'BUREAU_B_DEBT_PER_DAY_STD',
 'ACTIVE_BUREAU_B_DEBT_PER_DAY_MEAN',
 'CLOSED_BUREAU_B_REMAINING_TERM_MIN',
 'BUREAU_B_OVERDUE_TO_DAYS_STD',
 'AMT_REQ_CREDIT_BUREAU_WEEK',
 'docs_absent',
 'BUREAU_B_CLOSED_LATE_SUM',
 'mortgage_max_overdue_to_typedebt',
 'REG_CITY_NOT_WORK_CITY',
 'BUREAU_B_OVERDUE_TO_DAYS_MEAN',
 'ACTIVE_BUREAU_B_OVERDUE_TO_DAYS_MEAN',
 'BUREAU_B_DEBT_PER_DAY_MEAN',
 'BUREAU_B_RECENT_30D_MEAN',
 'ACTIVE_BUREAU_BB_LATEST_MON_MAX',
 'BUREAU_B_CREDIT_X_PROLONGED_MEAN',
 'CLOSED_BUREAU_B_RECENT_180D_MEAN',
 'FONDKAPREMONT_MODE',
 'BUREAU_BB_HISTORY_AGE_STD',
 'BUREAU_B_REMAINING_TERM_MIN',
 'BUREAU_BB_LATEST_MON_MAX',
 'CLOSED_BUREAU_B_DEBT_TO_TENURE_STD',
 'ACTIVE_BUREAU_B_DEBT_PER_DAY_MAX',
 'CLOSED_BUREAU_B_CLOSED_LATE_SUM',
 'BUREAU_B_DEBT_PER_DAY_MAX',
 'ACTIVE_BUREAU_B_CURRENT_TO_MAX_OVERDUE_STD',
 'CLOSED_BUREAU_BB_SMA0_SUM',
 'CLOSED_BUREAU_B_LEFT_TO_FULL_MEAN',
 'ACTIVE_BUREAU_BB_SMA0_MEAN',
 'CLOSED_BUREAU_B_DEBT_TO_DAYS_STD',
 'ACTIVE_BUREAU_BB_MAX_DEFAULT_MEAN',
 'CLOSED_BUREAU_BB_SMA1_RATIO_STD',
 'CLOSED_BUREAU_BB_SMA1_RATIO_MEAN',
 'CLOSED_BUREAU_BB_SMA1_MEAN',
 'BUREAU_B_DEBT_X_PROLONGED_MEAN',
 'BUREAU_CNT_CREDIT_PROLONG_STD',
 'FLAG_DOCUMENT_8',
 'CLOSED_BUREAU_B_LEFT_TO_FULL_MAX',
 'CLOSED_BUREAU_BB_HISTORY_AGE_STD',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN',
 'CLOSED_BUREAU_B_RECENT_90D_MEAN',
 'total_consumer_credit_overdue',
 'BUREAU_B_DEBT_X_PROLONGED_SUM',
 'CLOSED_BUREAU_B_LEFT_TO_FULL_STD',
 'BUREAU_B_CREDIT_X_PROLONGED_SUM',
 'CLOSED_BUREAU_AMT_CREDIT_SUM_DEBT_STD',
 'ACTIVE_BUREAU_BB_HISTORY_AGE_STD',
 'CLOSED_BUREAU_B_DEBT_TO_TENURE_MEAN',
 'BUREAU_B_CREDIT_X_PROLONGED_MAX',
 'ACTIVE_BUREAU_B_ACTIVE_DEBT_MAX',
 'ACTIVE_BUREAU_B_ACTIVE_DEBT_SUM',
 'CLOSED_BUREAU_AMT_CREDIT_SUM_LIMIT_MAX',
 'credit_card_max_overdue_to_total',
 'AMT_REQ_CREDIT_BUREAU_DAY',
 'REG_REGION_NOT_WORK_REGION',
 'ACTIVE_BUREAU_B_CREDIT_X_CURR_DPD_MEAN',
 'ACTIVE_BUREAU_BB_SMA0_SUM',
 'CLOSED_BUREAU_AMT_CREDIT_SUM_DEBT_MEAN',
 'BUREAU_B_CLOSED_DEBT_SUM',
 'BUREAU_B_CURR_OVERDUE_TO_DEBT_STD',
 'ACTIVE_BUREAU_B_RECENT_30D_MEAN',
 'ACTIVE_BUREAU_B_DEBT_X_PROLONGED_SUM',
 'BUREAU_CNT_CREDIT_PROLONG_MEAN',
 'BUREAU_B_CREDIT_X_CURR_DPD_MEAN',
 'BUREAU_BB_SMA1_RATIO_MEAN',
 'BUREAU_BB_SMA1_RATIO_STD',
 'BUREAU_B_OVERDUE_X_CURR_DPD_MEAN',
 'BUREAU_BB_CURR_STAT_MEAN',
 'FLAG_EMAIL',
 'BUREAU_B_LOG_CURR_DPD_MEAN',
 'CLOSED_BUREAU_BB_SMA1_RATIO_MAX',
 'BUREAU_B_CLOSED_DEBT_MEAN',
 'HOUSETYPE_MODE',
 'ACTIVE_BUREAU_B_EXPIRED_LOAN_SUM',
 'BUREAU_B_RECENT_90D_SUM',
 'ACTIVE_BUREAU_BB_CURR_STAT_MEAN',
 'ACTIVE_BUREAU_B_OVERDUE_TO_TENURE_MEAN',
 'CLOSED_BUREAU_BB_SMA0_MAX',
 'BUREAU_B_CURR_OVERDUE_TO_DEBT_MEAN',
 'BUREAU_B_OVERDUE_TO_TENURE_MEAN',
 'BUREAU_B_OVERDUE_TO_DAYS_MAX',
 'ACTIVE_BUREAU_B_OVERDUE_TO_DAYS_STD',
 'BUREAU_BB_NPA_RATIO_STD',
 'BUREAU_B_CURRENT_TO_MAX_OVERDUE_STD',
 'BUREAU_CNT_CREDIT_PROLONG_MAX',
 'BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN',
 'BUREAU_B_CURR_OVERDUE_TO_LOAN_MEAN',
 'ACTIVE_BUREAU_B_OVERDUE_TO_DAYS_MAX',
 'LIVE_REGION_NOT_WORK_REGION',
 'total_car_loan_max_overdue',
 'BUREAU_BB_SMA1_RATIO_MAX',
 'BUREAU_B_DEBT_X_PROLONGED_MAX',
 'FLAG_DOCUMENT_5',
 'car_loan_max_overdue_to_typedebt',
 'consumer_credit_overdue_to_typedebt',
 'CLOSED_BUREAU_B_RECENT_365D_SUM',
 'BUREAU_B_OVERDUE_TO_TENURE_MAX',
 'FLAG_OWN_REALTY_BIN',
 'BUREAU_B_OVERDUE_TO_TENURE_STD',
 'BUREAU_BB_SMA1_MEAN',
 'CLOSED_BUREAU_B_RECENT_180D_SUM',
 'ACTIVE_BUREAU_B_CREDIT_X_PROLONGED_SUM',
 'ACTIVE_BUREAU_CNT_CREDIT_PROLONG_MAX',
 'EMERGENCYSTATE_MODE',
 'CLOSED_BUREAU_AMT_CREDIT_SUM_LIMIT_SUM',
 'ACTIVE_BUREAU_B_HAS_PROLONG_MEAN',
 'BUREAU_B_LOG_CURR_DPD_MAX',
 'BUREAU_AMT_CREDIT_SUM_OVERDUE_STD',
 'total_credit_card_overdue',
 'ACTIVE_BUREAU_B_RECENT_90D_SUM',
 'BUREAU_BB_SMA1_SUM',
 'BUREAU_B_CURR_OVERDUE_TO_LOAN_STD',
 'ACTIVE_BUREAU_B_CURR_OVERDUE_TO_DEBT_STD',
 'BUREAU_B_CURR_OVERDUE_TO_LOAN_MAX',
 'BUREAU_B_CLOSED_DEBT_MAX',
 'CLOSED_BUREAU_B_DEBT_TO_TENURE_MAX',
 'credit_card_overdue_to_typedebt',
 'total_microloan_max_overdue',
 'BUREAU_BB_MAX_DEFAULT_MAX',
 'total_car_loan_count',
 'CLOSED_BUREAU_BB_SMA2_RATIO_STD',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_OVERDUE_STD',
 'BUREAU_CREDIT_DAY_OVERDUE_MEAN',
 'CLOSED_BUREAU_CNT_CREDIT_PROLONG_MEAN',
 'ACTIVE_BUREAU_B_CURR_OVERDUE_TO_LOAN_STD',
 'CLOSED_BUREAU_BB_CURR_STAT_MEAN',
 'BUREAU_B_CURR_OVERDUE_TO_DEBT_MAX',
 'CLOSED_BUREAU_B_DEBT_TO_DAYS_MEAN',
 'CLOSED_BUREAU_B_HAS_ACTUAL_END_SUM',
 'ACTIVE_BUREAU_B_MAX_OVERDUE_X_CURR_DPD_MEAN',
 'BUREAU_B_HAS_CURR_DPD_MEAN',
 'BUREAU_B_HAS_PROLONG_MEAN',
 'ACTIVE_BUREAU_B_OVERDUE_TO_TENURE_STD',
 'BUREAU_B_ACTIVE_OVERDUE_MEAN',
 'CLOSED_BUREAU_BB_SMA2_RATIO_MAX',
 'ACTIVE_BUREAU_B_HAS_ACTUAL_END_MEAN',
 'BUREAU_CREDIT_DAY_OVERDUE_STD',
 'ACTIVE_BUREAU_B_CREDIT_X_CURR_DPD_SUM',
 'CLOSED_BUREAU_B_CURRENT_TO_MAX_OVERDUE_STD',
 'FLAG_DOCUMENT_13',
 'CLOSED_BUREAU_B_DEBT_TO_DAYS_MAX',
 'REG_REGION_NOT_LIVE_REGION',
 'ACTIVE_BUREAU_B_CURRENT_TO_MAX_OVERDUE_MEAN',
 'BUREAU_B_HAS_OVERDUE_MEAN',
 'ACTIVE_BUREAU_CREDIT_DAY_OVERDUE_STD',
 'BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM',
 'ext_missing',
 'BUREAU_B_CREDIT_X_CURR_DPD_SUM',
 'CLOSED_BUREAU_AMT_CREDIT_SUM_DEBT_MAX',
 'total_mortgage_max_overdue',
 'BUREAU_BB_SMA2_MEAN',
 'ACTIVE_BUREAU_BB_SMA1_RATIO_STD',
 'ACTIVE_BUREAU_BB_SMA1_RATIO_MAX',
 'ACTIVE_BUREAU_B_DEBT_X_PROLONGED_MAX',
 'CLOSED_BUREAU_B_RECENT_30D_MEAN',
 'BUREAU_B_MULTI_PROLONG_MEAN',
 'ACTIVE_BUREAU_B_LOG_CURR_DPD_STD',
 'CLOSED_BUREAU_B_MAX_OVERDUE_TO_DEBT_MEAN',
 'CLOSED_BUREAU_BB_NPA_RATIO_STD',
 'ACTIVE_BUREAU_B_CREDIT_X_PROLONGED_MAX',
 'CLOSED_BUREAU_B_CREDIT_X_PROLONGED_MAX',
 'BUREAU_B_CURRENT_TO_MAX_OVERDUE_MEAN',
 'BUREAU_IS_SOLD_SUM',
 'BUREAU_BB_SMA2_MAX',
 'ACTIVE_BUREAU_CNT_CREDIT_PROLONG_STD',
 'mortgage_max_overdue_to_total',
 'BUREAU_BB_SMA2_RATIO_STD',
 'CLOSED_BUREAU_BB_SMA2_RATIO_MEAN',
 'FLAG_DOCUMENT_6',
 'ACTIVE_BUREAU_B_CURR_OVERDUE_TO_DEBT_MAX',
 'BUREAU_B_DEBT_X_CURR_DPD_MAX',
 'CLOSED_BUREAU_B_CREDIT_X_PROLONGED_MEAN',
 'ACTIVE_BUREAU_CNT_CREDIT_PROLONG_MEAN',
 'ACTIVE_BUREAU_CREDIT_DAY_OVERDUE_MEAN',
 'BUREAU_B_OVERDUE_X_CURR_DPD_SUM',
 'ACTIVE_BUREAU_B_CLOSED_EARLY_MEAN',
 'CLOSED_BUREAU_BB_NPA_RATIO_MAX',
 'BUREAU_B_DEBT_X_CURR_DPD_MEAN',
 'CLOSED_BUREAU_BB_NPA_MEAN',
 'ACTIVE_BUREAU_B_CURR_OVERDUE_TO_LOAN_MAX',
 'CLOSED_BUREAU_B_HAS_PROLONG_MEAN',
 'CLOSED_BUREAU_CNT_CREDIT_PROLONG_STD',
 'CLOSED_BUREAU_B_CREDIT_X_PROLONGED_SUM',
 'ACTIVE_BUREAU_B_DEBT_X_CURR_DPD_MEAN',
 'CLOSED_BUREAU_BB_NPA_MAX',
 'ACTIVE_BUREAU_B_LOG_CURR_DPD_MAX',
 'ACTIVE_BUREAU_B_OVERDUE_TO_TENURE_MAX',
 'total_mortgage_count',
 'BUREAU_BB_NPA_RATIO_MEAN',
 'CLOSED_BUREAU_BB_MAX_DEFAULT_MAX',
 'BUREAU_B_RECENT_30D_SUM',
 'CLOSED_BUREAU_B_DEBT_TO_CREDIT_STD',
 'ACTIVE_BUREAU_B_OVERDUE_X_CURR_DPD_SUM',
 'BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX',
 'BUREAU_BB_SMA2_RATIO_MEAN',
 'ACTIVE_BUREAU_B_OVERDUE_X_CURR_DPD_MEAN',
 'CLOSED_BUREAU_AMT_CREDIT_SUM_DEBT_SUM',
 'FLAG_DOCUMENT_18',
 'ACTIVE_BUREAU_B_CREDIT_X_CURR_DPD_MAX',
 'ACTIVE_BUREAU_B_LOG_CURR_DPD_MEAN',
 'microloan_max_overdue_to_typedebt',
 'ORG_IS_RARE',
 'ACTIVE_BUREAU_B_CURR_OVERDUE_TO_DEBT_MEAN',
 'BUREAU_B_MAX_OVERDUE_X_CURR_DPD_MEAN',
 'CLOSED_BUREAU_B_RECENT_90D_SUM',
 'BUREAU_BB_SMA2_RATIO_MAX',
 'consumer_credit_overdue_to_total',
 'ACTIVE_BUREAU_B_DEBT_X_CURR_DPD_MAX',
 'BUREAU_B_OVERDUE_PER_DAY_STD',
 'CLOSED_BUREAU_B_MAX_OVERDUE_TO_DEBT_MAX',
 'ACTIVE_BUREAU_B_OVERDUE_TO_MAX_OVERDUE_STD',
 'CLOSED_BUREAU_BB_SMA2_MEAN',
 'CLOSED_BUREAU_B_DEBT_PER_DAY_STD',
 'ACTIVE_BUREAU_B_CURR_OVERDUE_TO_LOAN_MEAN',
 'BUREAU_B_OVERDUE_TO_CREDIT_MEAN',
 'FLAG_DOCUMENT_11',
 'ACTIVE_BUREAU_B_OVERDUE_PER_DAY_MAX',
 'CLOSED_BUREAU_BB_SMA1_SUM',
 'ACTIVE_BUREAU_BB_SMA2_RATIO_MEAN',
 'BUREAU_B_CREDIT_X_CURR_DPD_MAX',
 'ACTIVE_BUREAU_CNT_CREDIT_PROLONG_MIN',
 'BUREAU_B_DPD30_FLAG_MEAN',
 'ACTIVE_BUREAU_B_OVERDUE_PER_DAY_MEAN',
 'CLOSED_BUREAU_B_DEBT_TO_CREDIT_MAX',
 'CLOSED_BUREAU_BB_NPA_RATIO_MEAN',
 'BUREAU_B_OVERDUE_PER_DAY_MAX',
 'BUREAU_B_DEBT_X_CURR_DPD_SUM',
 'BUREAU_B_MAX_OVERDUE_X_CURR_DPD_SUM',
 'BUREAU_B_MAX_OVERDUE_X_CURR_DPD_MAX',
 'BUREAU_B_OVERDUE_PER_DAY_MEAN',
 'ACTIVE_BUREAU_B_HAS_CURR_DPD_MEAN',
 'CLOSED_BUREAU_B_DEBT_TO_CREDIT_MEAN',
 'ACTIVE_BUREAU_B_ACTIVE_OVERDUE_MEAN',
 'CLOSED_BUREAU_B_CLOSED_DEBT_MEAN',
 'BUREAU_BB_NPA_MAX',
 'ACTIVE_BUREAU_B_HAS_OVERDUE_MEAN',
 'ACTIVE_BUREAU_B_OVERDUE_X_CURR_DPD_MAX',
 'CLOSED_BUREAU_BB_MISSING_MEAN',
 'BUREAU_BB_NPA_MEAN',
 'ACTIVE_BUREAU_BB_SMA1_MEAN',
 'BUREAU_B_ACTIVE_OVERDUE_SUM',
 'ACTIVE_BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX',
 'credit_card_overdue_to_total',
 'BUREAU_BB_CURR_STAT_MAX',
 'ACTIVE_BUREAU_B_OVERDUE_TO_DEBT_STD',
 'BUREAU_BB_MISSING_MEAN',
 'total_microloan_overdue',
 'FLAG_OWN_CAR_BIN',
 'BUREAU_B_ACTIVE_OVERDUE_MAX',
 'ACTIVE_BUREAU_B_RECENT_30D_SUM',
 'CLOSED_BUREAU_BB_CURR_STAT_MAX',
 'ACTIVE_BUREAU_BB_SMA1_RATIO_MEAN',
 'ACTIVE_BUREAU_B_DPD30_FLAG_MEAN']

In [5]:
str_cols = train_df[top_features_750].select_dtypes(
    include=['object', 'string']
).columns

print(str_cols)

for col in str_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

X=train_df[top_features_750]
y = train_df[TARGET_COL]
X_test=test_df[top_features_750]

Index(['CODE_GENDER', 'FLAG_OWN_REALTY', 'FLAG_OWN_CAR'], dtype='str')


In [6]:

# ============================================================
# CATEGORICAL FEATURES
# ============================================================

cat_cols = X.select_dtypes(
    include=['category']
).columns.tolist()

print(f"Categorical Features: {len(cat_cols)}")


Categorical Features: 17


In [ ]:


# ============================================================
# LIGHTGBM PARAMETERS
# ============================================================
params = {
    'objective': 'binary',
    'metric': 'auc',

    'boosting_type': 'gbdt',

    'learning_rate': 0.02,
    'n_estimators': 15000,

    'num_leaves': 128,
    'max_depth': -1,

    'min_child_samples': 50,

    'subsample': 0.9,
    'subsample_freq': 1,

    'colsample_bytree': 0.9,

    'reg_alpha': 0.1,
    'reg_lambda': 1.0,

    'random_state': 42,
    'n_jobs': -1,

    'verbosity': -1
}



In [8]:

# ============================================================
# CV SETUP
# ============================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)



In [9]:

# ============================================================
# STORAGE
# ============================================================

oof_preds = np.zeros(len(train_df))

test_preds = np.zeros(len(test_df))

cv_scores = []

feature_importance = pd.DataFrame()


In [10]:


# ============================================================
# TRAINING LOOP
# ============================================================
# original parameters

for fold, (train_idx, valid_idx) in enumerate(
        skf.split(X, y), 1):

    print("=" * 60)
    print(f"FOLD {fold}")
    print("=" * 60)

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    model = lgb.LGBMClassifier(
        **params
    )

    model.fit(
        X_train,
        y_train,

        eval_set=[
            (X_valid, y_valid)
        ],

        eval_metric='auc',

        categorical_feature=cat_cols,

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=300,
                verbose=True
            ),
            lgb.log_evaluation(
                period=200
            )
        ]
    )

    # ========================================================
    # VALIDATION PREDICTIONS
    # ========================================================

    valid_preds = model.predict_proba(
        X_valid,
        num_iteration=model.best_iteration_
    )[:, 1]

    oof_preds[valid_idx] = valid_preds

    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    fold_test_preds = model.predict_proba(
        X_test,
        num_iteration=model.best_iteration_
    )[:, 1]

    test_preds += fold_test_preds / N_SPLITS

    # ========================================================
    # FOLD SCORE
    # ========================================================

    fold_auc = roc_auc_score(
        y_valid,
        valid_preds
    )

    cv_scores.append(fold_auc)

    print(
        f"Fold {fold} AUC = {fold_auc:.6f}"
    )

    # ========================================================
    # FEATURE IMPORTANCE
    # ========================================================

    fold_importance = pd.DataFrame({
        'feature': top_features_750,
        'importance': model.feature_importances_,
        'fold': fold
    })

    feature_importance = pd.concat(
        [feature_importance, fold_importance],
        axis=0,
        ignore_index=True
    )

    del (
        X_train,
        X_valid,
        y_train,
        y_valid,
        valid_preds,
        fold_test_preds
    )

    gc.collect()
    


FOLD 1
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.769006
[400]	valid_0's auc: 0.775295
[600]	valid_0's auc: 0.77682
[800]	valid_0's auc: 0.777312
[1000]	valid_0's auc: 0.777081
Early stopping, best iteration is:
[769]	valid_0's auc: 0.777532
Fold 1 AUC = 0.777532
FOLD 2
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.768918
[400]	valid_0's auc: 0.774079
[600]	valid_0's auc: 0.774341
[800]	valid_0's auc: 0.775031
[1000]	valid_0's auc: 0.77416
Early stopping, best iteration is:
[788]	valid_0's auc: 0.775069
Fold 2 AUC = 0.775069
FOLD 3
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.780626
[400]	valid_0's auc: 0.787046
[600]	valid_0's auc: 0.788155
[800]	valid_0's auc: 0.788528
[1000]	valid_0's auc: 0.787939
Early stopping, best iteration is:
[773]	valid_0's auc: 0.78877
Fold 3 AUC = 0.788770
FOLD 4
Training until validation scores don't improve for 300 rounds
[200]	valid_0

In [13]:

# ============================================================
# OVERALL CV SCORE
# ============================================================

overall_auc = roc_auc_score(
    y,
    oof_preds
)

print("\n")
print("=" * 60)
print("CROSS VALIDATION RESULTS")
print("=" * 60)

for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score:.6f}")

print("\n")

print(
    f"Mean CV AUC : {np.mean(cv_scores):.6f}"
)

print(
    f"Std CV AUC  : {np.std(cv_scores):.6f}"
)

print(
    f"OOF AUC      : {overall_auc:.6f}"
)





CROSS VALIDATION RESULTS
Fold 1: 0.777532
Fold 2: 0.775069
Fold 3: 0.788770
Fold 4: 0.782508
Fold 5: 0.776948
Fold 6: 0.773293
Fold 7: 0.786074
Fold 8: 0.780931
Fold 9: 0.772825
Fold 10: 0.778183


Mean CV AUC : 0.779213
Std CV AUC  : 0.005042
OOF AUC      : 0.778991


1. n_folds=10 and 750 features<br>
Fold 1: 0.776549<br>
Fold 2: 0.775605<br>
Fold 3: 0.788007<br>
Fold 4: 0.781971<br>
Fold 5: 0.778129<br>
Fold 6: 0.774441<br>
Fold 7: 0.786221<br>
Fold 8: 0.781842<br>
Fold 9: 0.773873<br>
Fold 10: 0.779227<br>
Mean CV AUC : 0.779587<br>
Std CV AUC  : 0.004606<br>
OOF AUC      : 0.779394<br>

2. n_folds=10 and 850 features
Fold 1: 0.776830<br>
Fold 2: 0.775747<br>
Fold 3: 0.787616<br>
Fold 4: 0.782210<br>
Fold 5: 0.778598<br>
Fold 6: 0.774648<br>
Fold 7: 0.786353<br>
Fold 8: 0.780836<br>
Fold 9: 0.774180<br>
Fold 10: 0.777120<br>
Mean CV AUC : 0.779414<br>
Std CV AUC  : 0.004488<br>
OOF AUC      : 0.779151<br>

3. reg_alpha = 0.0 and reg_lambda=0.5 , n_folds = 10 and 800 features
Fold 1: 0.777532<br>
Fold 2: 0.775069<br>
Fold 3: 0.788770<br>
Fold 4: 0.782508<br>
Fold 5: 0.776948<br>
Fold 6: 0.773293<br>
Fold 7: 0.786074<br>
Fold 8: 0.780931<br>
Fold 9: 0.772825<br>
Fold 10: 0.778183<br>
Mean CV AUC : 0.779213<br>
Std CV AUC  : 0.005042<br>
OOF AUC      : 0.778991

In [14]:

# ============================================================
# FEATURE IMPORTANCE
# ============================================================

feature_importance_mean = (
    feature_importance
    .groupby('feature')['importance']
    .mean()
    .reset_index()
    .sort_values(
        by='importance',
        ascending=False
    )
)

print("\nTop 50 Features")
print(
    feature_importance_mean.head(50)
)




Top 50 Features
                                   feature  importance
713                      ORGANIZATION_TYPE      4662.0
751                      credit_to_annuity      2114.5
767                        goods_to_credit      1367.1
656                           EXT_SOURCE_3      1063.8
654                           EXT_SOURCE_1       971.5
643                 DAYS_LAST_PHONE_CHANGE       906.8
207                            AMT_ANNUITY       897.9
766                                ext_std       870.0
737                      annuity_to_income       829.7
655                           EXT_SOURCE_2       822.4
711                        OCCUPATION_TYPE       811.3
642                        DAYS_ID_PUBLISH       807.3
759                  ext_mean_x_employment       769.8
757                               ext_mean       754.1
762                                ext_min       752.5
769                   id_publish_age_ratio       738.5
640                             DAYS_BIRTH      